# 04c — ML vs LLM Consolidated Comparison  ⟨SCAFFOLD / TODO⟩

**This is a plan, not a finished analysis.** It is the intended *final* synthesis
notebook: one side-by-side of XGBoost vs the chosen GPT-5.4 config across every axis
the brief asks for (AUC, F1, stability, cost) plus explainability. Each section below
states **what it shows** and **which file feeds it**, with a `# TODO` stub to fill in.

The component analyses already exist — this notebook just consolidates them:
- `ml_models/04_Model_Analysis.ipynb` — XGBoost performance, calibration, SHAP, error.
- `01_model_selection/01g_Model_Analysis.ipynb` — GPT-5.4 reliance + error vs XGBoost.
- `02_prompt_variance/02d_Model_Analysis.ipynb` — same lens across prompt variants.

**Constraints (carry over):** no API calls, no retraining, **`test_batch` never loaded
here** (that is `04b` only). Runs on `tuning_sample`; prefer `04b` test metrics for the
headline scorecard *once `04b` has been run*.

## Build order / status checklist

- [ ] **§1 Headline scorecard** — one table, rows = axes, cols = XGBoost vs GPT-5.4 (no_desc) [+ GPT-5.4 high].
- [ ] **§2 Performance** — AUC / F1 / recall (Charged Off) bars.
- [ ] **§3 Calibration** — reliability curves + ECE/Brier, both sides ("scorer vs classifier").
- [ ] **§4 Feature reliance** — SHAP (XGB) vs surrogate (LLM) overlay + rank correlation.
- [ ] **§5 Error overlap** — do the two fail on the same loans? + portfolio error cost.
- [ ] **§6 Cost axis** — token cost vs credit-decision cost (the 01f insight: tokens ≈ rounding error).
- [ ] **§7 Explainability mode** — SHAP attributions vs LLM rationale/fingerprint.
- [ ] **§8 Executive verdict** — synthesise into the deck's one-slide takeaway.

**Upstream dependencies still pending:**
- [ ] `04b_Final_Benchmark.ipynb` not yet run → no `04b_final_benchmark.csv` (test-set numbers).
      Until then the scorecard rests on `tuning_sample` / robustness evidence — label it as such.
- [ ] Phase 3 hybrid not yet run → no `03_*` (ensemble column optional).

## Setup & input availability

Runnable check: confirms which result files this synthesis needs are present, so the
notebook doubles as a reminder of what is still missing.

In [1]:
import sys, os; sys.path.insert(0, '..')
import pandas as pd

RESULTS_LLM = '../../../data/results/llm'
RESULTS_ML  = '../../../data/results/ml'

# Each row: (axis it feeds, path). Presence drives the build checklist above.
INPUTS = {
    'XGBoost performance (test split)':   f'{RESULTS_ML}/03_model_performance.csv',
    'XGBoost feature importance (SHAP)':  f'{RESULTS_ML}/04_feature_importance.csv',
    'LLM model comparison (tuning)':      f'{RESULTS_LLM}/01a_metrics.csv',
    'LLM consistency / stability':        f'{RESULTS_LLM}/01b_metrics.csv',
    'LLM robustness (held-out batch)':    f'{RESULTS_LLM}/01c_metrics.csv',
    'LLM reasoning-effort sweep':         f'{RESULTS_LLM}/01d_metrics.csv',
    'LLM calibration (ECE/Brier)':        f'{RESULTS_LLM}/01e_confidence_metrics.csv',
    'LLM reasoning fingerprints':         f'{RESULTS_LLM}/01f_qualitative_financial.json',
    'Prompt-variant metrics':             f'{RESULTS_LLM}/02b_phase1_metrics.csv',
    'TEST benchmark (preferred headline)':f'{RESULTS_LLM}/04b_final_benchmark.csv',
    'Hybrid ensemble (optional column)':  f'{RESULTS_LLM}/03_blend_leaderboard.csv',
}
status = pd.DataFrame(
    [{'input': k, 'path': os.path.relpath(v), 'present': os.path.exists(v)} for k, v in INPUTS.items()])
print('Inputs present: {}/{}'.format(status['present'].sum(), len(status)))
display(status)
print('\nNOTE: if 04b_final_benchmark.csv is missing, build §1-§5 on tuning_sample and '
      'label results accordingly; swap in test metrics once 04b has run.')

Inputs present: 9/11


,input,path,present
0,XGBoost performance (test split),../../../data/results/ml/03_model_performance.csv,True
1,XGBoost feature importance (SHAP),../../../data/results/ml/04_feature_importance...,True
2,LLM model comparison (tuning),../../../data/results/llm/01a_metrics.csv,True
3,LLM consistency / stability,../../../data/results/llm/01b_metrics.csv,True
4,LLM robustness (held-out batch),../../../data/results/llm/01c_metrics.csv,True
5,LLM reasoning-effort sweep,../../../data/results/llm/01d_metrics.csv,True
6,LLM calibration (ECE/Brier),../../../data/results/llm/01e_confidence_metri...,True
7,LLM reasoning fingerprints,../../../data/results/llm/01f_qualitative_fina...,True
8,Prompt-variant metrics,../../../data/results/llm/02b_phase1_metrics.csv,True
9,TEST benchmark (preferred headline),../../../data/results/llm/04b_final_benchmark.csv,False



NOTE: if 04b_final_benchmark.csv is missing, build §1-§5 on tuning_sample and label results accordingly; swap in test metrics once 04b has run.


## §1 — Headline scorecard  `# TODO`

One table. Rows = Accuracy · AUC · F1 (CO) · Recall (CO) · Precision (CO) · Calibration
(ECE, Brier) · Error cost €/100 loans · Stability · Explainability mode. Columns =
**XGBoost** vs **GPT-5.4 (no_desc)** [+ GPT-5.4 high].

Sources: perf → `01a_metrics` (+ `03_model_performance` for XGB test split);
calibration → recompute XGB (see `04`) + `01e` for LLM; stability → `01b_metrics`
(std across 3 runs) + `01c_metrics` (robustness drop); error cost → recompute on
tuning sample (see `01g`).

In [2]:
# TODO §1: assemble the scorecard DataFrame (axes x {XGBoost, GPT-5.4}).
# Prefer 04b test metrics if present, else tuning-sample (01a) — label which.

## §2 — Performance  `# TODO`

AUC / F1 / recall (Charged Off) bars, XGBoost vs GPT-5.4. Source: `01a_metrics.csv`
(apples-to-apples, same 100 loans). Reference XGB test AUC ≈ 0.71 from
`03_model_performance.csv`.

In [3]:
# TODO §2: grouped bar chart of AUC / F1 / recall_charged_off.

## §3 — Calibration  `# TODO`

Reliability curve + ECE/Brier for both. The contrast: XGBoost emits a usable
probability; the LLM's token-confidence saturates (`01e`: gpt-5.4 ECE≈0.20).
**Caveat to verify:** XGB calibration on the 100-loan tuning sample (`04`) came out
ECE≈0.30 — noisy at n=100; recompute on `robustness_batch` before leaning on the
"better-calibrated scorer" framing.

Sources: recompute XGB (mirror `04` Part 2); LLM from `01e_confidence_metrics.csv`.

In [4]:
# TODO §3: overlay both reliability curves; tabulate ECE/Brier side by side.

## §4 — Feature reliance  `# TODO`

Overlay XGBoost SHAP importances vs GPT-5.4 surrogate reliance over the 30 source
features; report Spearman rank correlation. (In `01g` this came out ρ≈0.2 — low — i.e.
the two agree on outcomes far more than on *what they weight*.)

Sources: `04_feature_importance.csv` (XGB); recompute GPT surrogate (reuse `01g`
Part 2 method). Keep it generic over all features.

In [5]:
# TODO §4: load 04_feature_importance.csv, recompute GPT-5.4 surrogate importance,
# merge, plot side-by-side bars, print Spearman rank correlation.

## §5 — Error overlap  `# TODO`

Do XGBoost and GPT-5.4 fail on the **same** loans? Counts: wrong-by-both /
GPT-only / XGB-only (in `01g`: ~84% of GPT errors are also XGB errors). Plus portfolio
error cost per model (missed-default loss + false-rejection lost profit).

Source: `01a_predictions.csv` (has `llm_pred`, `xgb_pred`, `actual`).

In [6]:
# TODO §5: recompute error overlap + per-model error cost on tuning sample.

## §6 — Cost axis  `# TODO`

The brief's "cost" axis, reframed: API token cost vs credit-decision cost. The `01f`
ledger showed tokens (~€3 / 1000 loans) are ~5 orders of magnitude below credit P&L
(~€650k–940k). Make that scale contrast explicit.

Sources: `llm_calls.csv` (token cost) + the error-cost figures from §5 / `01f` ledger.

In [7]:
# TODO §6: bar/annotation contrasting token cost vs credit-decision cost.

## §7 — Explainability mode  `# TODO`

Qualitative contrast, not a metric: XGBoost gives **quantitative per-feature SHAP
attributions**; the LLM gives a **natural-language rationale** (and the `01f` reasoning
fingerprint). Show one SHAP waterfall next to one GPT-5.4 rationale for the same loan.

Sources: `04` waterfall; `01a_predictions.csv` `llm_reasoning`; `01f` fingerprint.

In [8]:
# TODO §7: side-by-side of a SHAP local explanation and the LLM's rationale text.

## §8 — Executive verdict  `# TODO`

One-paragraph synthesis for the deck: GPT-5.4 wins at its operating point (F1, error
cost) and reads like a human; XGBoost is the better-ranked scorer and is free; both are
grade/price-anchored on this dataset and share ~84% of their errors. State the honest
caveats (AUC gap, LLM calibration, low reliance-rank agreement).

In [9]:
# TODO §8: markdown verdict — fill once §1-§7 numbers are locked (and 04b is in).